In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

In [ ]:
KAGGLE_INPUT_BASE_PATH = "/kaggle/input/m5-forecasting-accuracy"
LOCAL_INPUT_BASE_PATH = "./data/raw/"
INPUT_BASE_PATH = LOCAL_INPUT_BASE_PATH

# Data constants
ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
N_EVENTS = 2
MAX_TRAIN_TIMESTAMP = 1941

EDA_RESULTS_PATH = Path("./results/eda/")
EDA_RESULTS_PATH.mkdir(parents=True, exist_ok=True)

## Load Data

In [ ]:
CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
SAMPLE_SUBMISSION = pl.read_csv(f"{INPUT_BASE_PATH}/sample_submission.csv")

In [ ]:
# Useful constants

STORE_IDS = SALES_TRAIN_EVALUATION["store_id"].unique().sort().to_list()
DEPT_IDS = SALES_TRAIN_EVALUATION["dept_id"].unique().sort().to_list()
ITEM_IDS = SALES_TRAIN_EVALUATION["item_id"].unique().sort().to_list()

## Basic Checks

In [ ]:
# Number of items per store_id / dept_id

item_ids_by_store_and_dept = (
    SALES_TRAIN_EVALUATION
    .select(["item_id", "dept_id", "store_id"])
    .group_by(["store_id", "dept_id"])
    .len(name="n_items")
    .sort(by=["store_id", "dept_id"])
)

store_ids = item_ids_by_store_and_dept["store_id"].unique().sort().to_list()
dept_ids = item_ids_by_store_and_dept["dept_id"].unique().sort().to_list()

fig, ax = plt.subplots(figsize=(14, 6))
cmap = plt.colormaps["viridis"]
colors = cmap(np.linspace(0.05, 0.95, len(store_ids)))

x = np.arange(len(dept_ids))
n_stores = len(store_ids)
bar_width = 0.8 / n_stores

# Common department order for aligning counts across stores
dept_df = pl.DataFrame({"dept_id": dept_ids})

for i, store_id in enumerate(store_ids):
    store_df = (
        item_ids_by_store_and_dept
        .filter(pl.col("store_id") == store_id)
        .select(["dept_id", "n_items"])
        .join(dept_df, on="dept_id", how="right")
        .sort("dept_id")
    )

    counts = (
        store_df["n_items"]
        .fill_null(0)
        .to_numpy()
    )

    offset = (i - (n_stores - 1) / 2) * bar_width

    ax.bar(
        x + offset,
        counts,
        width=bar_width,
        label=store_id,
        color=colors[i % len(colors)],
        edgecolor="white",
        linewidth=0.4,
        alpha=0.75,
    )

# Axis labels and title
ax.set_xticks(x)
ax.set_xticklabels(dept_ids, rotation=45, ha="right")
ax.set_ylabel("Number of items")
ax.set_title("Number of Items per Department and Store", pad=12)

# Legend and grid styling
ax.legend(title="Store ID", frameon=False, ncol=2, fontsize=9)
ax.grid(axis="y", alpha=0.25, linestyle="--")
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## Transform into long-form dataframe for analysis

In [ ]:
full_df = (
    SALES_TRAIN_EVALUATION.unpivot(
        index=ID_COLS,
        variable_name="d",
        value_name="sales",
    )
    .join(
        CALENDAR_DATA.select(pl.all().exclude("weekday", "wday", "month", "year")),
        on="d",
        how="left",
    )
    .join(
        SELL_PRICES,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
    )
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int64))
)

full_df.head()

## Sales Data Analysis

### Timeseries of sales for each department and store

In [ ]:
store_ids = full_df["store_id"].unique().sort().to_list()
dept_ids = full_df["dept_id"].unique().sort().to_list()

for store_id in store_ids:
    fig, axes = plt.subplots(len(dept_ids), 1, figsize=(10, len(dept_ids) * 2), sharex=True)
    for i, dept_id in enumerate(dept_ids):
        store_dept_df = (
            full_df
            .filter(pl.col("store_id") == store_id, pl.col("dept_id") == dept_id)
            .select(pl.col("item_id"), pl.col("d_index"), pl.col("sales"))

        )
        
        item_ids = store_dept_df["item_id"].unique().sort().to_list()
        for item_id in item_ids:
            item_df = (
                store_dept_df
                .filter(pl.col("item_id") == item_id)
                .sort(by=pl.col("d_index"))
            )
            axes[i].plot(
                item_df["d_index"].to_numpy(),
                item_df["sales"].to_numpy(),
                lw=0.25,
                color="grey",
                alpha=0.25,
            )

        avg_sales_df = (
            store_dept_df
            .group_by("d_index")
            .agg(
                pl.col("sales").mean().name.suffix("_mean"),
                pl.col("sales").quantile(0.5).name.suffix("_median"),
                pl.col("sales").quantile(0.95).name.suffix("_q95"),
                pl.col("sales").quantile(0.05).name.suffix("_q05")
            )
            .sort("d_index")
        )
        axes[i].plot(
            avg_sales_df["d_index"].to_numpy(),
            avg_sales_df["sales_mean"].to_numpy(),
            color="C0",
            lw=0.75,
        )
        
        axes[i].set(yscale="log", ylabel=dept_id)
        if i == len(dept_ids) - 1:
            axes[i].set(xlabel="Timestamp Index")

    fig.align_labels()
    fig.suptitle(store_id)
    fig.tight_layout();

    # plt.savefig(str(EDA_RESULTS_PATH / f"sales_timeseries_by_dept_store_{store_id}.png"), dpi=300);
    plt.close()

### Average sales by department for single store

In [ ]:
store_ids = full_df["store_id"].unique().sort().to_list()
dept_ids = full_df["dept_id"].unique().sort().to_list()

for store_id in store_ids:
    fig, axes = plt.subplots(len(dept_ids), 1, figsize=(12.5, len(dept_ids) * 1.5), sharex=True)
    for i, dept_id in enumerate(dept_ids):
        store_dept_df = (
            full_df
            .filter(pl.col("store_id") == store_id, pl.col("dept_id") == dept_id)
            .select(pl.col("item_id"), pl.col("d_index"), pl.col("sales"))
        )
        avg_sales_df = (
            store_dept_df
            .group_by("d_index")
            .agg(
                pl.col("sales").mean().name.suffix("_mean"),
                pl.col("sales").quantile(0.5).name.suffix("_median"),
                pl.col("sales").quantile(0.95).name.suffix("_q95"),
                pl.col("sales").quantile(0.05).name.suffix("_q05")
            )
            .sort("d_index")
        )
        axes[i].plot(
            avg_sales_df["d_index"].to_numpy(),
            avg_sales_df["sales_mean"].to_numpy(),
            color="C0",
            lw=0.75,
        )
        axes[i].grid(ls="--", alpha=0.5)
        axes[i].set(ylabel=dept_id)
        if i == len(dept_ids) - 1:
            axes[i].set(xlabel="Timestamp Index")

    fig.align_labels()
    fig.suptitle(store_id)
    fig.tight_layout();

    plt.savefig(str(EDA_RESULTS_PATH / f"avg_dept_sales_timeseries_store_{store_id}.png"), dpi=300);
    plt.close()

### Periods of zero sales where price is not null

In [ ]:
continuous_periods_of_zero_sales_df = (
    full_df
    .filter(
        pl.col("sales") == 0,
        pl.col("sell_price").is_not_null(),
    )
    .select("id", "item_id", "dept_id", "store_id", "d", "d_index", "date", "sell_price")
    .sort("id", "d_index")
    .with_columns(
        # cum_count increases incrementally with each row, while d_index
        # skips increments when there are non-zero sales. that gives each continuous
        # period of zero sales a unique period_id
        cum_count=pl.col("d_index").cum_count().over("id"),
        period_id=pl.col("d_index") - pl.col("d_index").cum_count().over(partition_by="id", order_by="d_index"),
    )
    .group_by("id", "item_id", "dept_id", "store_id", "period_id", maintain_order=True)
    .agg(
        pl.col("d_index").min().alias("start_d_index"),
        pl.col("d_index").max().alias("end_d_index"),
        pl.col("date").min().alias("start_date"),
        pl.col("date").max().alias("end_date"),
        pl.len().alias("n_days"),
    )
    .drop("period_id")
)

continuous_periods_of_zero_sales_df.head()

In [ ]:
# Plot distribution of zero-sale period lengths per dept_id / store_id.

min_period_days = 7
thresholded_zero_sale_periods = continuous_periods_of_zero_sales_df.filter(pl.col("n_days") > min_period_days)

store_ids = thresholded_zero_sale_periods["store_id"].unique().sort().to_list()
dept_ids = thresholded_zero_sale_periods["dept_id"].unique().sort().to_list()
_, bin_edges = np.histogram(thresholded_zero_sale_periods["n_days"].to_numpy(), bins=100)

for store_id in store_ids:
    store_periods_df = thresholded_zero_sale_periods.filter(pl.col("store_id") == store_id)

    fig, axes = plt.subplots(len(dept_ids), 1, figsize=(10, len(dept_ids) * 1.5), sharex=True)
    for i, dept_id in enumerate(dept_ids):
        n_days = (
            store_periods_df
            .filter(pl.col("dept_id") == dept_id)
            ["n_days"]
            .to_numpy()
        )
        axes[i].hist(n_days, bins=bin_edges, color="C0")
        axes[i].set(yscale="log", ylabel=dept_id, xlim=(0.5, None))
        axes[i].grid(ls="--", alpha=0.5)
        if i == len(dept_ids) - 1:
            axes[i].set(xlabel="Zero-sale period length (days)")

    fig.align_labels()
    fig.suptitle(f"{store_id} - min period = {min_period_days} days")
    fig.tight_layout()

    plt.savefig(str(EDA_RESULTS_PATH / f"zero_sale_period_length_by_dept_store_{store_id}.png"), dpi=300);
    plt.close();



In [ ]:
# Plot continuous zoer-sale periods.

min_period_days = 45
thresholded_zero_sale_periods = continuous_periods_of_zero_sales_df.filter(pl.col("n_days") > min_period_days)

samples_ids_to_plot = thresholded_zero_sale_periods["id"].sample(10)
fig, axes = plt.subplots(len(samples_ids_to_plot), 1, figsize=(10, len(samples_ids_to_plot) * 1.5), sharex=True)
for i, item_id in enumerate(samples_ids_to_plot):

    item_zero_sale_periods = (
        thresholded_zero_sale_periods
        .filter(pl.col("id") == item_id)
        .select(pl.col("start_d_index"), pl.col("end_d_index"))
    )
    for start_idx, end_idx in item_zero_sale_periods.iter_rows():
        sales_from_period = (
            full_df
            .filter(pl.col("id") == item_id, pl.col("d_index").is_between(start_idx, end_idx))
            .select(pl.col("id"), pl.col("item_id"), pl.col("d_index"), pl.col("sales"), pl.col("sell_price"))
            .sort(by=pl.col("d_index"))
        )
        assert sales_from_period["sell_price"].is_not_null().all()
        axes[i].scatter(
            sales_from_period["d_index"].to_numpy(),
            sales_from_period["sales"].to_numpy(),
            marker="x",
            s=5,
            color="tab:red",
            alpha=0.25,
        )
    
    # Plot full sales period
    full_item_df = (
        full_df
        .filter(pl.col("id") == item_id)
        .select(pl.col("id"), pl.col("item_id"), pl.col("d_index"), pl.col("sales"), pl.col("sell_price"))
        .sort(by=pl.col("d_index"))
    )
    sales_line = axes[i].plot(
        full_item_df["d_index"].to_numpy(),
        full_item_df["sales"].to_numpy(),
        lw=0.75,
        color="grey",
        alpha=0.75,
        label=item_id.rstrip("_evaluation"),
    )

    price_ax = axes[i].twinx()
    price_line = price_ax.plot(
        full_item_df["d_index"].to_numpy(),
        full_item_df["sell_price"].to_numpy(),
        lw=1.5,
        color="steelblue",
        alpha=0.75,
    )
    price_ax.set(ylabel="sell price")
    price_color = price_line[0].get_color()
    price_ax.yaxis.label.set_color(price_color)
    price_ax.tick_params(axis="y", colors=price_color)
    price_ax.spines["right"].set_color(price_color)

    axes[i].legend(loc=2, fontsize="small")
    axes[i].set(ylabel="sales")
    price_ax.set(ylim=(0, np.nanmax(full_item_df["sell_price"].to_numpy()) * 1.2))
    if i == len(samples_ids_to_plot) - 1: axes[i].set(xlabel="timestamp index")

fig.tight_layout();

### Seasonality of sales data

In [ ]:
# Seasonality by weekday only

sales_df_with_calendar_features = (
    full_df
    .select(["id", "dept_id", "store_id", "d_index", "date", "sales"])
    .with_columns(
        weekday=pl.col("date").dt.weekday(),
        week=pl.col("date").dt.week(),
        month=pl.col("date").dt.month(),
        year=pl.col("date").dt.year(),
    )
)

store_id = "WI_1"
store_df = sales_df_with_calendar_features.filter(pl.col("store_id") == store_id)
dept_ids = store_df["dept_id"].unique().sort().to_list()

fig, axes = plt.subplots(len(dept_ids), 1, figsize=(10, len(dept_ids) * 2), sharex=True)
for i, dept_id in enumerate(dept_ids):
    store_dept_df = store_df.filter(pl.col("dept_id") == dept_id)
    avg_sales_df = (
        store_dept_df
        .group_by(["id", "weekday"])
        .agg(avg_sales=pl.col("sales").mean())
        .sort(by=["id", "weekday"])
    )
    for item_id in avg_sales_df["id"].unique(maintain_order=True).to_list():
        item_df = avg_sales_df.filter(pl.col("id") == item_id)
        axes[i].plot(
            item_df["weekday"].to_numpy(),
            item_df["avg_sales"].to_numpy(),
            color="grey",
            lw=0.5,
            alpha=0.25,
        )

    # Average weekday pattern across every item and month in the store and department.
    avg_sales_by_weekday_df = (
        store_dept_df
        .group_by("weekday")
        .agg(avg_sales=pl.col("sales").mean())
        .sort("weekday")
    )
    axes[i].plot(
        avg_sales_by_weekday_df["weekday"].to_numpy(),
        avg_sales_by_weekday_df["avg_sales"].to_numpy(),
        color="C0",
        lw=1.75,
        zorder=3,
    )

    axes[i].set(ylabel=dept_id, yscale="log")
    axes[i].grid(ls="--", alpha=0.5)
    if i == len(dept_ids) - 1:
        axes[i].set(xlabel="weekday", xticks=range(1, 8))
        axes[i].set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])

fig.align_labels()
fig.suptitle(store_id)
fig.tight_layout()


In [ ]:
# Seasonality by weekday and month

sales_df_with_calendar_features = (
    full_df
    .select(["id", "dept_id", "store_id", "d_index", "date", "sales"])
    .with_columns(
        weekday=pl.col("date").dt.weekday(),
        week=pl.col("date").dt.week(),
        month=pl.col("date").dt.month(),
        year=pl.col("date").dt.year(),
    )
)

store_id = "TX_3"
store_df = sales_df_with_calendar_features.filter(pl.col("store_id") == store_id)
dept_ids = store_df["dept_id"].unique().sort().to_list()

month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
month_colors = plt.colormaps["viridis"].resampled(12)

fig, axes = plt.subplots(len(dept_ids), 1, figsize=(10, len(dept_ids) * 2), sharex=True)
for i, dept_id in enumerate(dept_ids):
    store_dept_df = store_df.filter(pl.col("dept_id") == dept_id)
    avg_sales_df = (
        store_dept_df
        .group_by(["id", "weekday"])
        .agg(avg_sales=pl.col("sales").mean())
        .sort(by=["id", "weekday"])
    )
    for item_id in avg_sales_df["id"].unique(maintain_order=True).to_list():
        item_df = avg_sales_df.filter(pl.col("id") == item_id)
        axes[i].plot(
            item_df["weekday"].to_numpy(),
            item_df["avg_sales"].to_numpy(),
            color="grey",
            lw=0.5,
            alpha=0.25,
        )

    # Average weekday pattern per month, across every item in the store and department.
    avg_sales_by_month_df = (
        store_dept_df
        .group_by(["month", "weekday"])
        .agg(avg_sales=pl.col("sales").mean())
        .sort(["month", "weekday"])
    )
    for month in range(1, 13):
        month_df = avg_sales_by_month_df.filter(pl.col("month") == month)
        axes[i].plot(
            month_df["weekday"].to_numpy(),
            month_df["avg_sales"].to_numpy(),
            color=month_colors(month - 1),
            lw=1.75,
            label=month_labels[month - 1] if i == 0 else None,
            zorder=3,
        )

    axes[i].set(ylabel=dept_id, yscale="log")
    axes[i].grid(ls="--", alpha=0.5)
    if i == len(dept_ids) - 1:
        axes[i].set(xlabel="weekday", xticks=range(1, 8))
        axes[i].set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])

axes[0].legend(loc="upper left", fontsize="small", ncol=2, title="month")
fig.align_labels()
fig.suptitle(store_id)
fig.tight_layout()

### Is a Poisson distribution a sensible choice? Dispersion and zero-inflation checks

1. **Marginal dispersion**: variance vs mean per series (Poisson: var = mean).
2. **Conditional dispersion**: Poisson GLM Pearson chi2 / df after conditioning on calendar effects (weekday, month, SNAP, events) and price. Marginal overdispersion can come from the mean varying with covariates, so this is the fairer test.
3. **Zero-inflation**: observed zeros vs zeros predicted by Poisson and negative binomial with the same covariates. Zero-inflation means excess zeros relative to the *negative binomial*, not relative to Poisson. Only days with a non-null price are used.

Fits are run on a random sample of series.

In [ ]:
import warnings

RANDOM_SEED = 0
N_SERIES_SAMPLE = 300
MIN_ACTIVE_DAYS = 365

# Only days where the item is on sale (price not null). Zeros before an item's
# first price are structural, not demand, and would look like zero-inflation.
active_df = (
    full_df
    .filter(pl.col("sell_price").is_not_null())
    .with_columns(
        log_sales=pl.col("sales").log1p(),
        weekday=pl.col("date").dt.weekday(),
        month=pl.col("date").dt.month(),
        snap=(
            pl.when(pl.col("state_id") == "CA").then(pl.col("snap_CA"))
            .when(pl.col("state_id") == "TX").then(pl.col("snap_TX"))
            .otherwise(pl.col("snap_WI"))
        ),
        has_event=pl.col("event_name_1").is_not_null().cast(pl.Int8),
        log_price=pl.col("sell_price").log(),
    )
    .select(
        "id", "store_id", "dept_id", "d_index", "sales", "log_sales",
        "weekday", "month", "snap", "has_event", "log_price"
    )
)

series_stats = (
    active_df
    .group_by("id", "dept_id", "store_id")
    .agg(
        n_days=pl.len(),
        mean=pl.col("sales").mean(),
        var=pl.col("sales").var(),
        zero_frac=(pl.col("sales") == 0).mean(),
    )
    .filter(pl.col("n_days") >= MIN_ACTIVE_DAYS, pl.col("mean") > 0)
    .with_columns(dispersion_index=pl.col("var") / pl.col("mean"))
)

# Marginal check: variance vs mean. Poisson => var == mean.

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
m = series_stats["mean"].to_numpy()
v = series_stats["var"].to_numpy()
axes[0].scatter(m, v, s=4, alpha=0.3, color="C0")
lims = [min(m.min(), v[v > 0].min()), max(m.max(), v.max())]
axes[0].plot(lims, lims, color="tab:red", lw=1, label="var = mean (Poisson)")
axes[0].set(xscale="log", yscale="log", xlabel="mean daily sales", ylabel="variance of daily sales")
axes[0].legend(frameon=False)
axes[0].grid(ls="--", alpha=0.5)

di = series_stats["dispersion_index"].to_numpy()
axes[1].hist(di, bins=np.logspace(np.log10(max(di.min(), 1e-2)), np.log10(di.max()), 60), color="C0")
axes[1].axvline(1, color="tab:red", lw=1, label="Poisson")
axes[1].set(xscale="log", xlabel="dispersion index (var / mean)", ylabel="number of series")
axes[1].legend(frameon=False)
axes[1].grid(ls="--", alpha=0.5)
fig.suptitle("Marginal dispersion (no covariates)")
fig.tight_layout()

print(f"{(di > 1).mean():.1%} of {len(di)} series have var/mean > 1")


In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

sample_ids = (
    series_stats["id"].sample(min(N_SERIES_SAMPLE, series_stats.height), seed=RANDOM_SEED)
    .to_list()
)
sample_pd = active_df.filter(pl.col("id").is_in(sample_ids)).sort("id", "d_index").to_pandas()
sample_by_id = {k: g.reset_index(drop=True) for k, g in sample_pd.groupby("id")}

FORMULAS = {
    "intercept only": "sales ~ 1",
    "+ calendar": "sales ~ C(weekday) + C(month) + snap + has_event",
    "+ calendar + price": "sales ~ C(weekday) + C(month) + snap + has_event + log_price",
}

# Pearson chi2 / residual df of a Poisson GLM: ~1 if Poisson holds *given the covariates*,
# >> 1 means genuine overdispersion that covariates do not explain.
rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for series_id, g in sample_by_id.items():
        row = {"id": series_id}
        for name, formula in FORMULAS.items():
            try:
                res = smf.glm(formula, data=g, family=sm.families.Poisson()).fit()
                row[name] = res.pearson_chi2 / res.df_resid
            except Exception:
                row[name] = np.nan
        rows.append(row)
conditional_dispersion = pl.DataFrame(rows).join(series_stats.select("id", "mean"), on="id")

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.logspace(-0.5, 1.5, 50)
for j, name in enumerate(FORMULAS):
    vals = conditional_dispersion[name].drop_nulls().to_numpy()
    ax.hist(
        vals, bins=bins, histtype="step", lw=1.75, color=f"C{j}",
        label=f"{name} (median {np.median(vals):.2f})"
    )
ax.axvline(1, color="tab:red", lw=1, ls="--")
ax.set(xscale="log", xlabel="Poisson Pearson dispersion (chi2 / df)", ylabel="number of series")
ax.legend(frameon=False)
ax.grid(ls="--", alpha=0.5)
ax.set_title("Dispersion before / after conditioning on covariates")
fig.tight_layout()

conditional_dispersion.select(pl.col(list(FORMULAS)).median())


In [ ]:
# Zero-inflation: compare observed zeros with the zeros predicted by Poisson and NB
# fitted with the same covariates. Zero-inflation = zeros in EXCESS of what the NB predicts.
FORMULA = FORMULAS["+ calendar + price"]
N_PMF_MAX = 10  # counts 0..N_PMF_MAX-1 compared in the observed-vs-expected check

def longest_zero_run(y):
    best = cur = 0
    for v in y:
        cur = cur + 1 if v == 0 else 0
        best = max(best, cur)
    return best

fits = {}
rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for series_id, g in sample_by_id.items():
        y = g["sales"].to_numpy()
        try:
            pois = smf.glm(FORMULA, data=g, family=sm.families.Poisson()).fit()
            nb = smf.negativebinomial(FORMULA, data=g).fit(disp=0, maxiter=200)
            mu_p = pois.predict(g).to_numpy()
            mu_nb = nb.predict(g).to_numpy()
            alpha = float(nb.params["alpha"])
        except Exception:
            continue
        if not np.isfinite(alpha) or alpha <= 0 or not np.isfinite(mu_nb).all():
            continue

        # Calculate P(y=0)
        p0_pois = np.exp(-mu_p)
        p0_nb = (1.0 / (1.0 + alpha * mu_nb)) ** (1.0 / alpha)
        fits[series_id] = {"y": y, "mu_p": mu_p, "mu_nb": mu_nb, "alpha": alpha}
        rows.append({
            "id": series_id,
            "n_days": len(y),
            "obs_zeros": int((y == 0).sum()),
            "exp_zeros_poisson": p0_pois.sum(),
            "exp_zeros_nb": p0_nb.sum(),
            "alpha": alpha,
            "longest_zero_run": longest_zero_run(y),
            "loglik_poisson": pois.llf,
            "loglik_nb": nb.llf,
        })

zero_check = (
    pl.DataFrame(rows)
    .join(series_stats.select("id", "mean", "dept_id"), on="id")
    .with_columns(
        excess_zeros_poisson=(pl.col("obs_zeros") - pl.col("exp_zeros_poisson")) / pl.col("n_days"),
        excess_zeros_nb=(pl.col("obs_zeros") - pl.col("exp_zeros_nb")) / pl.col("n_days"),
    )
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# (1) observed vs expected zeros
for j, (col, name) in enumerate([("exp_zeros_poisson", "Poisson"), ("exp_zeros_nb", "NegBin")]):
    axes[0].scatter(
        zero_check[col] / zero_check["n_days"],
        zero_check["obs_zeros"] / zero_check["n_days"],
        s=6, alpha=0.4, color=f"C{j}", label=name
    )
axes[0].plot([0, 1], [0, 1], color="grey", lw=1)
axes[0].set(xlabel="expected zero fraction", ylabel="observed zero fraction")
axes[0].legend(frameon=False)
axes[0].grid(ls="--", alpha=0.5)

# (2) excess zeros (observed - expected, as a fraction of days)
bins = np.linspace(-0.2, 0.5, 60)
for j, (col, name) in enumerate([("excess_zeros_poisson", "Poisson"), ("excess_zeros_nb", "NegBin")]):
    axes[1].hist(
        zero_check[col].to_numpy(), bins=bins, histtype="step", lw=1.75, color=f"C{j}",
        label=f"{name} (median {zero_check[col].median():+.3f})"
    )
axes[1].axvline(0, color="grey", lw=1, ls="--")
axes[1].set(xlabel="(observed - expected zeros) / n_days", ylabel="number of series")
axes[1].legend(frameon=False)
axes[1].grid(ls="--", alpha=0.5)

# (3) are zeros clustered in long runs (stockout / delisting) rather than independent?
axes[2].scatter(zero_check["excess_zeros_nb"], zero_check["longest_zero_run"], s=6, alpha=0.4, color="C0")
axes[2].set(yscale="log", xlabel="NegBin excess zero fraction", ylabel="longest run of zeros (days)")
axes[2].grid(ls="--", alpha=0.5)

fig.suptitle("Zero-inflation check (active days only)")
fig.tight_layout()

zero_check.select(
    pl.col("excess_zeros_poisson").median().alias("median_excess_poisson"),
    pl.col("excess_zeros_nb").median().alias("median_excess_nb"),
    (pl.col("excess_zeros_nb") > 0.05).mean().alias("frac_series_nb_excess_gt_5pct"),
    (pl.col("loglik_nb") - pl.col("loglik_poisson")).median().alias("median_loglik_gain_nb"),
)


In [ ]:
# Observed vs expected count frequencies for example series, spanning low to high volume.
# expected count of k = sum over days of P(y_i = k | mu_i).
ranked = zero_check.sort("mean")
example_ids = [ranked["id"][int(q * (ranked.height - 1))] for q in (0.1, 0.25, 0.5, 0.7, 0.8, 0.9, 0.95)]

ks = np.arange(15)
fig, axes = plt.subplots(1, len(example_ids), figsize=(4 * len(example_ids), 3.5), sharey=False)
for ax, series_id in zip(axes, example_ids):
    f = fits[series_id]
    # Count number of days for each k 
    obs = np.array([(f["y"] == k).sum() for k in ks])

    # Calculate expected number of k's given fitted distributions.
    exp_p = np.array([stats.poisson.pmf(k, f["mu_p"]).sum() for k in ks])
    
    n_r = 1.0 / f["alpha"]
    exp_nb = np.array([stats.nbinom.pmf(k, n_r, n_r / (n_r + f["mu_nb"])).sum() for k in ks])

    # Plot and compare
    ax.bar(ks, obs, color="lightgrey", label="observed")
    ax.plot(ks, exp_p, marker="o", ms=4, color="C0", lw=1.5, label="Poisson")
    ax.plot(ks, exp_nb, marker="o", ms=4, color="C1", lw=1.5, label="NegBin")
    ax.set(yscale="log", xlabel="daily sales", title=f"{series_id.removesuffix('_evaluation')}\nmean={f['y'].mean():.2f}")
    ax.grid(ls="--", alpha=0.5)
axes[0].set(ylabel="days")
axes[0].legend(frameon=False, fontsize="small")
fig.tight_layout()


In [ ]:
# Estimate the Tweedie variance power rho from Var(y | mu) = phi * mu^rho.
# Idea: use mu from the Poisson GLM (consistent for the mean even if overdispersed),
# bin days by mu, and regress log(mean squared residual) on log(mu). The slope is rho
# (1 = Poisson, 2 = gamma-like; negative binomial lies in between).
N_MU_BINS = 25

mu_all = np.concatenate([f["mu_p"] for f in fits.values()])
sq_resid_all = np.concatenate([(f["y"] - f["mu_p"]) ** 2 for f in fits.values()])

def estimate_variance_power(mu, sq_resid, n_bins=N_MU_BINS, min_count=50):
    """Binned log-log regression. Returns (rho, log_phi, bin_mu, bin_var)."""
    edges = np.quantile(mu, np.linspace(0, 1, n_bins + 1))
    edges = np.unique(edges)
    idx = np.clip(np.digitize(mu, edges[1:-1]), 0, len(edges) - 2)
    bin_mu, bin_var, w = [], [], []
    for b in range(len(edges) - 1):
        mask = idx == b
        if mask.sum() < min_count:
            continue
        bin_mu.append(mu[mask].mean())
        bin_var.append(sq_resid[mask].mean())
        w.append(mask.sum())
    bin_mu, bin_var, w = map(np.asarray, (bin_mu, bin_var, w))
    rho, log_phi = np.polyfit(np.log(bin_mu), np.log(bin_var), 1, w=np.sqrt(w))
    return rho, log_phi, bin_mu, bin_var

rho, log_phi, bin_mu, bin_var = estimate_variance_power(mu_all, sq_resid_all)

# Bootstrap over series for an uncertainty estimate on the pooled rho.
rng = np.random.default_rng(RANDOM_SEED)
series_keys = list(fits)
boot_rhos = []
for _ in range(200):
    pick = rng.choice(len(series_keys), size=len(series_keys), replace=True)
    mu_b = np.concatenate([fits[series_keys[i]]["mu_p"] for i in pick])
    sq_b = np.concatenate([(fits[series_keys[i]]["y"] - fits[series_keys[i]]["mu_p"]) ** 2 for i in pick])
    boot_rhos.append(estimate_variance_power(mu_b, sq_b)[0])
rho_lo, rho_hi = np.percentile(boot_rhos, [2.5, 97.5])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(bin_mu, bin_var, s=25, color="C0", label="binned data")
grid = np.logspace(np.log10(bin_mu.min()), np.log10(bin_mu.max()), 100)
axes[0].plot(grid, np.exp(log_phi) * grid ** rho, color="C1", lw=1.75,
             label=f"fit: rho = {rho:.2f} [{rho_lo:.2f}, {rho_hi:.2f}]")
axes[0].plot(grid, grid, color="grey", lw=1, ls="--", label="Poisson (rho = 1)")
axes[0].plot(grid, grid ** 2 * bin_var[-1] / bin_mu[-1] ** 2, color="grey", lw=1, ls=":", label="rho = 2 (anchored)")
axes[0].set(xscale="log", yscale="log", xlabel="fitted mean mu", ylabel="mean squared residual")
axes[0].legend(frameon=False, fontsize="small")
axes[0].grid(ls="--", alpha=0.5)

# Does rho depend on volume? Estimate within low / mid / high volume series tertiles.
series_mean = {k: f["y"].mean() for k, f in fits.items()}
cuts = np.quantile(list(series_mean.values()), [1 / 3, 2 / 3])
group_rhos = {}
for name, lo, hi in [("low volume", -np.inf, cuts[0]), ("mid volume", cuts[0], cuts[1]), ("high volume", cuts[1], np.inf)]:
    keys = [k for k, m_ in series_mean.items() if lo < m_ <= hi]
    mu_g = np.concatenate([fits[k]["mu_p"] for k in keys])
    sq_g = np.concatenate([(fits[k]["y"] - fits[k]["mu_p"]) ** 2 for k in keys])
    group_rhos[name] = estimate_variance_power(mu_g, sq_g)[0]
axes[1].bar(list(group_rhos), list(group_rhos.values()), color="C0", width=0.5)
axes[1].axhline(1, color="grey", lw=1, ls="--")
axes[1].axhline(2, color="grey", lw=1, ls=":")
axes[1].set(ylabel="estimated rho", ylim=(0, 2.2))
axes[1].grid(axis="y", ls="--", alpha=0.5)

fig.suptitle("Tweedie variance power: Var(y) = phi * mu^rho")
fig.tight_layout()

print(f"pooled rho = {rho:.2f}  (95% bootstrap CI over series: [{rho_lo:.2f}, {rho_hi:.2f}])")
print({k: round(float(v), 2) for k, v in group_rhos.items()})


## Price data analysis

### Are periods of missing prices single continuous periods?

In [ ]:
no_price_df = full_df.filter(pl.col("sell_price").is_null())
items_ids_no_price = no_price_df["item_id"].unique().sort().to_list()

item_ids_with_non_cont_missing_prices = []
for item_id in items_ids_no_price:
    item_df = (
        no_price_df
        .filter(pl.col("item_id") == item_id)
        .sort(by=pl.col("d_index"))
        .select(pl.col("d_index").diff().alias("d_index_diff"))
        .filter((~pl.col("d_index_diff").is_null()), pl.col("d_index_diff").gt(1))
    )
    if not item_df.is_empty():
        item_ids_with_non_cont_missing_prices.append(item_id)


len(item_ids_with_non_cont_missing_prices)

### Are sales non-zero when price is null?

In [ ]:
assert full_df.filter(pl.col("sell_price").is_null())["sales"].eq(0).all()

full_df.filter(pl.col("sell_price").is_null(), pl.col("sales").gt(0))

In [ ]:
# Inspect timeseries of sales for when prices are null.

items_ids_no_price = no_price_df["id"].unique().sort().to_list()

start_idx = 1500
n_items_to_plot = 10
item_ids_to_plot = items_ids_no_price[int(start_idx * n_items_to_plot): int((start_idx + 1) * n_items_to_plot)]

fig, axes = plt.subplots(len(item_ids_to_plot), 1, figsize=(10, len(item_ids_to_plot) * 1.5), sharex=True)
for i, item_id in enumerate(item_ids_to_plot):
    
    # Plot sales for period where price is null
    no_price_item_df = (
        no_price_df
        .filter(pl.col("id") == item_id)
        .select(pl.col("id"), pl.col("item_id"), pl.col("d_index"), pl.col("sales"))
        .sort(by=pl.col("d_index"))
    )
    axes[i].scatter(
        no_price_item_df["d_index"].to_numpy(),
        no_price_item_df["sales"].to_numpy(),
        marker="x",
        s=5,
        color="tab:red",
        alpha=0.25,
    )
    
    # Plot full sales period
    full_item_df = (
        full_df
        .filter(pl.col("id") == item_id)
        .select(pl.col("id"), pl.col("item_id"), pl.col("d_index"), pl.col("sales"))
        .sort(by=pl.col("d_index"))
    )
    axes[i].plot(
        full_item_df["d_index"].to_numpy(),
        full_item_df["sales"].to_numpy(),
        lw=0.75,
        color="grey",
        alpha=0.75,
        label=item_id.rstrip("_evaluation")
    )

    axes[i].legend(loc=1)
    axes[i].set(ylabel="sales")
    if i == len(item_ids_to_plot) - 1: axes[i].set(xlabel="timestamp index")

fig.align_labels()
fig.tight_layout();

In [ ]:
# Check that for each item without sell price, the period without prices
# is at the start of the sales timeseries.

items_ids_no_price = no_price_df["id"].unique().sort().to_list()
for item_id in items_ids_no_price:
    no_price_item_df = (
        no_price_df
        .filter(pl.col("id") == item_id)
        .select(pl.col("id"), pl.col("item_id"), pl.col("d_index"), pl.col("sales"), pl.col("sell_price"))
        .sort(by=pl.col("d_index"))
    )
    assert no_price_item_df["d_index"].min() == 1
    assert no_price_item_df["d_index"].max() == len(no_price_item_df)
    assert (no_price_item_df["d_index"] == pl.arange(1, len(no_price_item_df) + 1, eager=True, dtype=pl.UInt64).alias("d_index")).all()


In [ ]:
# Is price ever 0.0?

price_is_zero_df = full_df.filter(pl.col("sell_price") == 0.0)
price_is_zero_df.head()

### Min / Max timestamps per item id after removing null prices

In [ ]:
min_max_dates_by_id = (
    full_df
    .filter(~pl.col("sell_price").is_null())
    .group_by(["id", "dept_id", "store_id"])
    .agg(min_date=pl.col("date").min(), max_date=pl.col("date").max())
    .sort(["store_id", "dept_id", "min_date"])
)

store_ids = min_max_dates_by_id["store_id"].unique().sort().to_list()
for store_id in store_ids:
    dept_ids = (
        min_max_dates_by_id.filter(pl.col("store_id") == store_id)
        ["dept_id"]
        .unique()
        .sort()
        .to_list()
    )

    fig, axes = plt.subplots(len(dept_ids), 1, figsize=(10, len(dept_ids) * 2), sharex=True)
    for d, dept_id in enumerate(dept_ids):
        item_ids = (
            min_max_dates_by_id
            .filter(pl.col("store_id") == store_id, pl.col("dept_id") == dept_id)
            .sort(by="min_date")
            ["id"]
            .to_list()
        )
        
        for i, item_id in enumerate(item_ids):
            item_df = (
                min_max_dates_by_id
                .filter(
                    pl.col("store_id") == store_id,
                    pl.col("dept_id") == dept_id,
                    pl.col("id") == item_id
                )
            )
            min_date, max_date = next(item_df[["min_date", "max_date"]].iter_rows())
            if i % 2 == 0:
                axes[d].plot(
                    [min_date, max_date],
                    [i, i],
                    color="grey",
                    lw=len(item_ids) / 800,
                    alpha=0.5
                )
            axes[d].scatter(
                [min_date],
                [i],
                marker="o",
                color="steelblue",
                s=15,
                label="min_date" if i == 0 else None,
                alpha=0.75
            )
            axes[d].scatter(
                [max_date],
                [i],
                marker="o",
                color="darkorange",
                s=15,
                label="max_date" if i == 0 else None,
                alpha=0.75
            )

        axes[d].legend(fontsize="small")
        axes[d].set(ylabel="ItemID", title=f"{store_id}, {dept_id}")
    
    fig.align_labels()
    fig.tight_layout();

    plt.savefig(str(EDA_RESULTS_PATH / f"min_max_item_id_timestamp_by_dept_store_{store_id}.png"), dpi=300);
    plt.close()

### Number of training observations per item id after removing null prices

In [ ]:
n_samples_per_item_dept_store = (
    full_df
    .filter(~pl.col("sell_price").is_null())
    .select(["id", "dept_id", "store_id", "date"])
    .unique()
    .group_by(["id", "dept_id", "store_id"])
    .len(name="n_observations")
)

n_samples_per_item_dept_store.head()

## Forecast Period Analysis

In [ ]:
# Is price null in the forecasting period?
CALENDAR_DATA.head()

In [ ]:
prices_and_dates = (
    CALENDAR_DATA
    .select(["date", "wm_yr_wk", "d"])
    .join(
        other=SELL_PRICES,
        on="wm_yr_wk",
        how="left",
    )
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int64))
)

In [ ]:
(
    prices_and_dates
    .group_by("item_id")
    .agg(
        d_index_min=pl.col("d_index").min(),
        d_index_max=pl.col("d_index").max(),
    )
    .filter(pl.col("d_index_max") < 1969)
)

In [ ]:
len(SALES_TRAIN_EVALUATION["item_id"].unique())

In [ ]:
(
    CALENDAR_DATA
    .filter(pl.col("date") == CALENDAR_DATA["date"].max())
)